# ETC — Results Analysis

Evaluates the atomicity of ETC's predicted hunk groupings/commits against ground truth labels using the following three metrics:
- **Rand Index** (RI) — pairwise agreement
- **Adjusted Rand Index** (ARI) — corrects for bias, via sklearn
- **Perfect Match Rate** (PMR) — fraction of GT groups exactly reproduced

In [36]:
import csv
from pathlib import Path

import pandas as pd
from sklearn.metrics import adjusted_rand_score

In [37]:
# ── Configuration ─────────────────────────────────────────────────────────────
def _find_autocommit_dir() -> Path:
    """Locate the autocommit/ directory regardless of the notebook's working dir.

    Works whether the kernel's cwd is the repo root or the autocommit/ folder.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "autocommit" / "results" / "runs.csv").exists():
            return base / "autocommit"
        if base.name == "autocommit" and (base / "results" / "runs.csv").exists():
            return base
    raise FileNotFoundError(f"Could not locate autocommit/ from cwd={Path.cwd()}")


AUTOCOMMIT_DIR = _find_autocommit_dir()
TESTS_DIR      = AUTOCOMMIT_DIR / "tests"
RESULTS_DIR    = AUTOCOMMIT_DIR / "results"

# Which test case to evaluate against
TEST_NAME = "889ac3bf1-75hunks"
TEST_DIR  = TESTS_DIR / TEST_NAME

# Run IDs to evaluate (from autocommit/results/runs.csv)
RUN_IDS = ["20260626_141536", "20260628_221153", "20260628_233151", "20260608_115913", "20260628_222124", "20260629_113705"]


## Patch loading and hunk matching

In [38]:
def _changed_content(hunk_body: list[str]) -> tuple:
    """Return the +/- content lines of a hunk, excluding the diff metadata lines."""
    return tuple(
        line[1:]
        for line in hunk_body
        if line.startswith(("+", "-")) and not line.startswith(("---", "++"+"+"))
    )


def extract_hunk_fingerprints(patch_text: str) -> list[tuple[str, tuple]]:
    """Parse a unified diff and return one fingerprint per hunk."""
    fingerprints: list[tuple[str, tuple]] = []
    current_file = ""
    hunk_body: list[str] = []
    in_hunk = False

    for line in patch_text.splitlines():
        if line.startswith("diff --git "):
            if in_hunk:
                fingerprints.append((current_file, _changed_content(hunk_body)))
            current_file = line.split(" b/", 1)[1]
            hunk_body = []
            in_hunk = False
        elif line.startswith("@@"):
            if in_hunk:
                fingerprints.append((current_file, _changed_content(hunk_body)))
            hunk_body = []
            in_hunk = True
        elif in_hunk:
            hunk_body.append(line)

    if in_hunk:
        fingerprints.append((current_file, _changed_content(hunk_body)))

    return fingerprints

In [39]:
def load_ground_truth(gt_dir: Path) -> dict[tuple, int]:
    """Parse all gt-commit*.patch / gt_commit*.patch files in gt_dir."""
    fp_to_label: dict[tuple, int] = {}
    gt_files = sorted(
        set(gt_dir.glob("gt-commit*.patch")) | set(gt_dir.glob("gt_commit*.patch"))
    )
    print(f"Ground truth: {[f.name for f in gt_files]}")

    for label, gt_file in enumerate(gt_files):
        fps = extract_hunk_fingerprints(gt_file.read_text())
        for fp in fps:
            if fp in fp_to_label:
                print(f"  Warning: fingerprint appears in multiple GT files — {fp[0]}")
            fp_to_label[fp] = label
        print(f"  {gt_file.name} → label {label}, {len(fps)} hunks")

    return fp_to_label


fp_to_gt = load_ground_truth(TEST_DIR)

Ground truth: ['gt-commit1.patch', 'gt-commit2.patch', 'gt-commit3.patch']
  gt-commit1.patch → label 0, 35 hunks
  gt-commit2.patch → label 1, 20 hunks
  gt-commit3.patch → label 2, 21 hunks


In [40]:
def find_run_dir(run_id: str) -> Path | None:
    """Locate the run directory for a given run_id."""
    for test_subdir in TESTS_DIR.iterdir():
        candidate = test_subdir / "runs" / run_id
        if candidate.exists():
            return candidate
    candidate = AUTOCOMMIT_DIR / "runs" / run_id
    return candidate if candidate.exists() else None


def load_predicted_grouping(run_id: str) -> list[tuple[tuple, int]]:
    """
    Load hunk files for a run and return a list of (content_fp, group_num) pairs.

    Using a list rather than a dict so that hunk files with identical content
    (duplicate changes at different positions) are kept as separate entries.
    A dict would silently collapse them via key overwrite.
    """
    run_dir = find_run_dir(run_id)
    if run_dir is None:
        raise FileNotFoundError(f"Run directory not found for run_id={run_id!r}")
    hunks_dir = run_dir / "hunks"

    hunk_to_group: dict[str, int] = {}
    with open(RESULTS_DIR / "groups.csv") as f:
        for row in csv.DictReader(f):
            if row["run_id"] != run_id:
                continue
            group_num = int(row["group_num"])
            for name in row["hunks"].split(", "):
                hunk_to_group[name.strip()] = group_num

    pred_list: list[tuple[tuple, int]] = []
    for hunk_name, group_num in hunk_to_group.items():
        hunk_file = hunks_dir / hunk_name
        if not hunk_file.exists():
            print(f"  Warning: missing hunk file {hunk_name}")
            continue
        fps = extract_hunk_fingerprints(hunk_file.read_text())
        if not fps:
            print(f"  Warning: no fingerprint from {hunk_name}")
            continue
        pred_list.append((fps[0], group_num))

    n_groups = len({g for _, g in pred_list})
    print(f"Run {run_id}: {n_groups} groups, {len(pred_list)} committed hunks")
    return pred_list


In [41]:
def build_label_vectors(
    fp_to_gt: dict[tuple, int],
    pred_list: list[tuple[tuple, int]],
) -> tuple[list[int], list[int]]:
    """
    Produce parallel y_true / y_pred vectors by matching each predicted hunk
    to the ground truth via its content fingerprint.
    Hunks with no GT match are excluded; GT hunks with no predicted match are reported.
    """
    pred_fps = {fp for fp, _ in pred_list}
    only_pred_fps = pred_fps - set(fp_to_gt)   # committed but fingerprint not in GT
    only_gt_fps   = set(fp_to_gt) - pred_fps   # in GT but never committed

    y_true: list[int] = []
    y_pred: list[int] = []
    for content_fp, group_num in pred_list:
        if content_fp in fp_to_gt:
            y_true.append(fp_to_gt[content_fp])
            y_pred.append(group_num)

    print(f"Matched hunks            : {len(y_true)}")
    print(f"Committed but not in GT  : {len([fp for fp, _ in pred_list if fp in only_pred_fps])}")
    for fp in sorted(only_pred_fps):
        print(f"  {fp[0]}")
    print(f"In GT but not committed  : {len(only_gt_fps)}")
    for fp in sorted(only_gt_fps):
        print(f"  {fp[0]}")

    return y_true, y_pred


## Rand Index

For every pair of hunks $(i, j)$, both clusterings either agree or disagree on whether they belong together:

$$RI = \frac{TP + TN}{TP + TN + FP + FN}$$

| | Same in $\hat{y}$ | Different in $\hat{y}$ |
|---|---|---|
| **Same in $y$** | TP | FN |
| **Different in $y$** | FP | TN |

In [42]:
def rand_index(y_true: list[int], y_pred: list[int]) -> dict:
    n = len(y_true)
    tp = tn = fp = fn = 0
    for i in range(n):
        for j in range(i + 1, n):
            same_true = y_true[i] == y_true[j]
            same_pred = y_pred[i] == y_pred[j]
            if same_true and same_pred:
                tp += 1
            elif not same_true and not same_pred:
                tn += 1
            elif same_pred:   # not same_true
                fp += 1
            else:             # same_true, not same_pred
                fn += 1
    total = tp + tn + fp + fn
    ri = (tp + tn) / total if total > 0 else 0.0
    return {"RI": ri, "TP": tp, "TN": tn, "FP": fp, "FN": fn, "total_pairs": total}

## Adjusted Rand Index

Raw RI can be misleadingly high when one cluster dominates — e.g. if ETC produces 74 singleton
groups and 1 large group, the vast majority of pairs are correctly identified as *different*,
inflating TN even when the grouping is substantially wrong.  The Adjusted Rand Index corrects
for this by subtracting the expected RI of a random clustering with the same cluster size distribution:

$$ARI = \frac{RI - E[RI]}{\max(RI) - E[RI]}$$

ARI = 1 is a perfect match; ARI ≈ 0 is no better than random; ARI < 0 is worse than random.
This gives a more honest picture of grouping quality when singleton outputs dominate the TN count.


In [43]:
def compute_ari(y_true: list[int], y_pred: list[int]) -> float:
    return adjusted_rand_score(y_true, y_pred)

## Perfect Match Rate

For each ground truth group, check whether any predicted group contains **exactly** the same
set of hunk fingerprints.  PMR = matched GT groups / total GT groups.

When no exact match exists, we also report the best Jaccard overlap as a diagnostic.

In [44]:
def perfect_match_rate(
    fp_to_gt: dict[tuple, int],
    pred_list: list[tuple[tuple, int]],
) -> dict:
    gt_groups: dict[int, set] = {}
    for fp, label in fp_to_gt.items():
        gt_groups.setdefault(label, set()).add(fp)

    pred_groups: dict[int, set] = {}
    for content_fp, group_num in pred_list:
        pred_groups.setdefault(group_num, set()).add(content_fp)

    pred_sets = list(pred_groups.values())

    matched = 0
    for gt_label, gt_set in sorted(gt_groups.items()):
        exact = any(gt_set == p for p in pred_sets)
        if exact:
            matched += 1
            print(f"  GT group {gt_label} ({len(gt_set)} hunks): EXACT MATCH")
        else:
            best_j = max(
                (len(gt_set & p) / len(gt_set | p) for p in pred_sets if gt_set | p),
                default=0.0,
            )
            print(f"  GT group {gt_label} ({len(gt_set)} hunks): no exact match, best Jaccard = {best_j:.3f}")

    pmr = matched / len(gt_groups) if gt_groups else 0.0
    return {"PMR": pmr, "matched": matched, "total_gt_groups": len(gt_groups)}


## Concern fusion rate

For each output commit, check whether its hunks span more than one GT group.
A fused commit contains changes from multiple concerns that ddmin could not separate
because they are build-interdependent — the primary failure mode of the programmatic approach.

Rate = fused output commits / total output commits with at least one GT-matched hunk.

In [45]:
def concern_fusion_rate(
    fp_to_gt: dict[tuple, int],
    pred_list: list[tuple[tuple, int]],
) -> dict:
    # For each group, collect the GT labels of its matched hunks
    group_gt_labels: dict[int, set[int]] = {}
    for content_fp, group_num in pred_list:
        if content_fp in fp_to_gt:
            group_gt_labels.setdefault(group_num, set()).add(fp_to_gt[content_fp])

    fused = {gn: labels for gn, labels in group_gt_labels.items() if len(labels) > 1}
    total = len(group_gt_labels)

    print(f"  Groups with matched hunks    : {total}")
    print(f"  Fused groups (>1 GT concern) : {len(fused)}")
    for gn, labels in sorted(fused.items()):
        print(f"    Group {gn}: spans GT groups {sorted(labels)}")

    rate = len(fused) / total if total > 0 else 0.0
    return {"concern_fusion_rate": rate, "fused_groups": len(fused), "total_groups": total}


runs_df = pd.read_csv(RESULTS_DIR / "runs.csv")
meta = runs_df[runs_df["run_id"] == RUN_ID].iloc[0]

print(f"Run: {RUN_ID}  approach={meta['approach']}  groups={meta['groups_produced']}")

pred_list = load_predicted_grouping(RUN_ID)

print("\n── Hunk matching ──")
y_true, y_pred = build_label_vectors(fp_to_gt, pred_list)

print("\n── Rand Index ──")
ri_result = rand_index(y_true, y_pred)
print(f"  RI = {ri_result['RI']:.4f}  "
      f"(TP={ri_result['TP']}, TN={ri_result['TN']}, "
      f"FP={ri_result['FP']}, FN={ri_result['FN']}, pairs={ri_result['total_pairs']})")

print("\n── Adjusted Rand Index ──")
ari = compute_ari(y_true, y_pred)
print(f"  ARI = {ari:.4f}")

print("\n── Perfect Match Rate ──")
pmr_result = perfect_match_rate(fp_to_gt, pred_list)
print(f"  PMR = {pmr_result['PMR']:.4f}  "
      f"({pmr_result['matched']}/{pmr_result['total_gt_groups']} GT groups exactly matched)")

print("\n── Concern Fusion Rate ──")
cfr_result = concern_fusion_rate(fp_to_gt, pred_list)
print(f"  CFR = {cfr_result['concern_fusion_rate']:.4f}  "
      f"({cfr_result['fused_groups']}/{cfr_result['total_groups']} groups fused)")

pd.DataFrame([{
    "run_id":               RUN_ID,
    "approach":             meta["approach"],
    "groups":               meta["groups_produced"],
    "invocations":          meta["total_invocations"],
    "matched_hunks":        len(y_true),
    "RI":                   round(ri_result["RI"], 4),
    "ARI":                  round(ari, 4),
    "PMR":                  round(pmr_result["PMR"], 4),
    "concern_fusion_rate":  round(cfr_result["concern_fusion_rate"], 4),
}]).set_index("run_id")


In [46]:
DRILL_RUN_ID = RUN_IDS[0]   # change to inspect a specific run in detail

runs_df = pd.read_csv(RESULTS_DIR / "runs.csv")
meta = runs_df[runs_df["run_id"] == DRILL_RUN_ID].iloc[0]
drill_test_dir = find_run_dir(DRILL_RUN_ID).parent.parent
gt = load_ground_truth(drill_test_dir)

print(f"\nRun: {DRILL_RUN_ID}  test_case={drill_test_dir.name}  "
      f"approach={meta['approach']}  groups={meta['groups_produced']}")

pred_list = load_predicted_grouping(DRILL_RUN_ID)

print("\n── Hunk matching ──")
y_true, y_pred = build_label_vectors(gt, pred_list)

print("\n── Rand Index ──")
ri_result = rand_index(y_true, y_pred)
print(f"  RI = {ri_result['RI']:.4f}  "
      f"(TP={ri_result['TP']}, TN={ri_result['TN']}, "
      f"FP={ri_result['FP']}, FN={ri_result['FN']}, pairs={ri_result['total_pairs']})")

print("\n── Adjusted Rand Index ──")
ari = compute_ari(y_true, y_pred)
print(f"  ARI = {ari:.4f}")

print("\n── Perfect Match Rate ──")
pmr_result = perfect_match_rate(gt, pred_list)
print(f"  PMR = {pmr_result['PMR']:.4f}  "
      f"({pmr_result['matched']}/{pmr_result['total_gt_groups']} GT groups exactly matched)")

print("\n── Concern Fusion Rate ──")
cfr_result = concern_fusion_rate(gt, pred_list)
print(f"  CFR = {cfr_result['concern_fusion_rate']:.4f}  "
      f"({cfr_result['fused_groups']}/{cfr_result['total_groups']} groups fused)")


Ground truth: ['gt-commit1.patch', 'gt-commit2.patch', 'gt-commit3.patch']
  gt-commit1.patch → label 0, 3 hunks
  gt-commit2.patch → label 1, 4 hunks
  gt-commit3.patch → label 2, 1 hunks

Run: 20260626_141536  test_case=1fe68f0d53-8hunks  approach=programmatic  groups=5
Run 20260626_141536: 5 groups, 8 committed hunks

── Hunk matching ──
Matched hunks            : 8
Committed but not in GT  : 0
In GT but not committed  : 0

── Rand Index ──
  RI = 0.8929  (TP=6, TN=19, FP=0, FN=3, pairs=28)

── Adjusted Rand Index ──
  ARI = 0.7308

── Perfect Match Rate ──
  GT group 0 (3 hunks): no exact match, best Jaccard = 0.333
  GT group 1 (4 hunks): EXACT MATCH
  GT group 2 (1 hunks): EXACT MATCH
  PMR = 0.6667  (2/3 GT groups exactly matched)

── Concern Fusion Rate ──
  Groups with matched hunks    : 5
  Fused groups (>1 GT concern) : 0
  CFR = 0.0000  (0/5 groups fused)


## Multi-run evaluation

Score every run in `RUN_IDS` against its own test case's ground truth, then report a single overall macro-average (each run weighted equally).

In [47]:
import io
import contextlib

# Ground truth is per-test-case; cache it so the same case is not reloaded.
_GT_CACHE: dict[Path, dict] = {}


def _ground_truth_for(test_dir: Path) -> dict:
    if test_dir not in _GT_CACHE:
        with contextlib.redirect_stdout(io.StringIO()):
            _GT_CACHE[test_dir] = load_ground_truth(test_dir)
    return _GT_CACHE[test_dir]


def evaluate_run(run_id: str, runs_df: pd.DataFrame) -> dict | None:
    """Score one run against its OWN test case's ground truth.

    Returns a row dict, or None (with a reason printed) if the run cannot be
    scored — e.g. its run dir or ground truth is missing, or no committed hunk
    matches the ground truth.
    """
    run_dir = find_run_dir(run_id)
    if run_dir is None:
        print(f"  {run_id}: run dir not found — skipped")
        return None
    test_dir = run_dir.parent.parent
    gt = _ground_truth_for(test_dir)
    if not gt:
        print(f"  {run_id}: no ground truth in {test_dir.name}/ — skipped")
        return None

    with contextlib.redirect_stdout(io.StringIO()):
        pred_list = load_predicted_grouping(run_id)
        y_true, y_pred = build_label_vectors(gt, pred_list)
    if not y_true:
        print(f"  {run_id}: no committed hunk matched {test_dir.name}/ ground truth — skipped")
        return None

    ri = rand_index(y_true, y_pred)
    ari = compute_ari(y_true, y_pred)
    with contextlib.redirect_stdout(io.StringIO()):
        pmr = perfect_match_rate(gt, pred_list)
        cfr = concern_fusion_rate(gt, pred_list)

    rows = runs_df[runs_df["run_id"] == run_id]
    meta = rows.iloc[0] if len(rows) else {}
    return {
        "run_id":        run_id,
        "test_case":     test_dir.name,
        "approach":      meta.get("approach", "?") if len(rows) else "?",
        "groups":        meta.get("groups_produced", float("nan")) if len(rows) else float("nan"),
        "invocations":   meta.get("total_invocations", float("nan")) if len(rows) else float("nan"),
        "matched_hunks": len(y_true),
        "RI":            round(ri["RI"], 4),
        "ARI":           round(ari, 4),
        "PMR":           round(pmr["PMR"], 4),
        "CFR":           round(cfr["concern_fusion_rate"], 4),
    }


In [48]:
runs_df = pd.read_csv(RESULTS_DIR / "runs.csv")

print(f"Evaluating {len(RUN_IDS)} run(s):")
rows = [r for rid in RUN_IDS if (r := evaluate_run(rid, runs_df)) is not None]
results_df = pd.DataFrame(rows).set_index("run_id")

# Overall macro-average (each scored run weighted equally) + std.
metric_cols = ["RI", "ARI", "PMR", "CFR"]
num_cols = ["groups", "invocations", "matched_hunks", *metric_cols]

print(f"\nScored {len(results_df)}/{len(RUN_IDS)} runs. Overall macro-average:")
for c in metric_cols:
    print(f"  {c}: {results_df[c].mean():.4f} ± {results_df[c].std():.4f}")

mean_row: dict[str, object] = {c: round(results_df[c].mean(), 4) for c in num_cols}
mean_row.update({"test_case": f"MEAN (n={len(results_df)})", "approach": ""})
results_df.loc["__mean__"] = [mean_row.get(c, "") for c in results_df.columns]

results_df


Evaluating 6 run(s):

Scored 6/6 runs. Overall macro-average:
  RI: 0.5864 ± 0.2391
  ARI: 0.1385 ± 0.2917
  PMR: 0.2500 ± 0.2935
  CFR: 0.0000 ± 0.0000


,test_case,approach,groups,invocations,matched_hunks,RI,ARI,PMR,CFR
run_id,,,,,,,,,
20260626_141536,1fe68f0d53-8hunks,programmatic,5.0000,43.0000,8.0000,0.8929,0.7308,0.6667,0.0
20260628_221153,18e5e7850-5hunks,programmatic,5.0000,13.0000,5.0000,0.8000,0.0000,0.3333,0.0
20260628_233151,465cbd0a8-49hunks,programmatic,41.0000,606.0000,49.0000,0.4753,0.0220,0.0000,0.0
20260608_115913,889ac3bf1-75hunks,programmatic,58.0000,3792.0000,74.0000,0.6679,0.0783,0.0000,0.0
20260628_222124,ac9e31c5a-13hunks,programmatic,13.0000,50.0000,13.0000,0.2821,0.0000,0.0000,0.0
20260629_113705,dd25e301a-5hunks,programmatic,5.0000,13.0000,5.0000,0.4000,0.0000,0.5000,0.0
__mean__,MEAN (n=6),,21.1667,752.8333,25.6667,0.5864,0.1385,0.2500,0.0
